In [ ]:
### Things learn during drils to add to sheat cheet 

# .all() / .any() in assert (see drill 2)

In [ ]:
### Drill 1: Question: for each room type, how many listings are there, and what share of them has no price (in %, rounded to 1 decimal place)? 
# Sort by that share, highest first.

import pandas as pd  
df = pd.read_parquet("../data/clean/listings_clean.parquet") 

table=(
    df
    .assign(missing_price=lambda d: d["price_eur"].isna())
    .groupby("room_type", as_index=False)
    .agg(n_listings=("id","size"), pct_no_price=("missing_price", "mean"))
    .assign(pct_no_price=lambda d: (d["pct_no_price"]*100).round(1))
    .sort_values("pct_no_price", ascending=False)
)
display(table)

# Sanity checks 
assert table["n_listings"].sum()==len(df)

# 15 minutes: I got stuck multiplying the mean of missing_price within the .agg so I ended up doing it later in the code

In [ ]:
### Drill 2 (target: 6 min)

import pandas as pd  
df = pd.read_parquet("../data/clean/listings_clean.parquet") 

# Question: among priced listings, which neighbourhoods have at least 1,000 listings? 
# For those, give the number of listings and the median price, and show the 5 most expensive by median.
table = (
    df 
    .loc[lambda d: d["price_eur"].notna()]
    .groupby("neighbourhood_cleansed", as_index=False)
    .agg(n_listings=("id","size"), median_price=("price_eur","median"))
    .loc[lambda d: d["n_listings"]>=1000]
    .sort_values("median_price", ascending=False)
    .head(5)
)
# Expected output: a table with 5 rows and 3 columns: neighbourhood_cleansed, n_listings, median_price.

# Constraint: write it as one chain. The “at least 1,000” filter must come after the aggregation (the HAVING pattern).

# Sanity check: every n_listings in your table is ≥ 1,000, written as an assert.
for n_listing in table["n_listings"]:
    assert n_listing>=1000 
assert (table["n_listings"] >= 1000).all() # -> table[...]>=1000 create a Series of True/False, then .all() check it for all rows

display(table)
# 16 minutes: got stuck at placing the .agg and .loc at the right place, and writing the sanity check: is the loop correct?

In [ ]:
### Drill 3 (target 6 min)

# Question: among priced listings, for each room type, give the number of listings, the mean price, the median price, 
# and the ratio of mean to median (rounded to 2 decimals). Sort by the ratio, highest first.

import pandas as pd  
df = pd.read_parquet("../data/clean/listings_clean.parquet") 

table=(
    df
    .loc[df["price_eur"].notna()]
    .groupby("room_type", as_index=False)
    .agg(
        n_listings = ("id","size"),
        mean_price = ("price_eur", "mean"),
        median_price = ("price_eur", "median")
    )
    .assign(mean_median_ratio = lambda d: d["mean_price"]/d["median_price"])
    .round({"mean_median_ratio":2, "mean_price":2, "median_price":2})
    .sort_values("mean_median_ratio", ascending=False)
)
display(table)

# Expected output: a table with 4 rows and 5 columns: room_type, n_listings, mean_price, median_price, mean_median_ratio.

# Constraint: one chain, with the ratio computed in an .assign after the .agg.

# Sanity check (as code): the sum of n_listings equals the number of priced listings, 43,537.
assert table["n_listings"].sum()==df["price_eur"].notna().sum()

# Then, in one sentence: which room type has the most outlier-driven mean, and what does that mean for reporting its price?
# Private rooms have the highest mean/median ratio (1.87, against 1.64 for shared room, 1.52 for entire homes, and 1.28 for hotels): the mean (€296) is pulled up by a few very expensive listings, 
# so I’d report the median (€158) and check the top prices for errors.

# 8min47 

In [ ]:
### Drill 4 (target 6 minutes)

# Question: among neighbourhoods with at least 3,000 listings (all room types, priced or not), 
# which 5 have the highest share of listings without a price (in %, 1 decimal)?

import pandas as pd  
df = pd.read_parquet("../data/clean/listings_clean.parquet") 

by_neigh = (
    df
    .assign(not_priced=lambda d: (d["price_eur"].isna()*100))
    .groupby("neighbourhood_cleansed", as_index=False)
    .agg(
        n_listings=("id","size"),
        pct_no_price=("not_priced","mean")
    )
)
assert by_neigh["n_listings"].sum()==len(df)

table = (
    by_neigh
    .loc[ lambda d: d["n_listings"]>=3000 ]
    .sort_values("pct_no_price", ascending=False)
    .assign(pct_no_price = lambda d: (d["pct_no_price"].round(1)))
    .head(5)
)
display(table)

# Expected output: up to 5 rows and 3 columns: neighbourhood_cleansed, n_listings, pct_no_price. 
# If fewer than 5 neighbourhoods pass the filter, say how many.

# Constraints:

# One chain.
# The percent is built before the agg, with the * 100 trick from yesterday.
# The 3,000 filter comes after the agg.

# Sanity check (as code): use .all() to check that every n_listings is ≥ 3,000. 
assert (table["n_listings"]>=3000).all()    #assert passes
# Then add one check that can fail: the sum of n_listings before the filter equals len(df). 
# This means you need to store the table before the filter in a separate variable.

In [ ]:
### Review drill A (target: 12 min, first attempt timed)

# Question: for priced listings, does a shared bathroom go with a lower price? Give the median price by room type (only Entire home/apt and Private room)
# and by bathroom type: shared, private, or unknown when the text is missing.

import pandas as pd  
df = pd.read_csv("../data/raw/listings.csv.gz", usecols=["id", "room_type", "bathrooms_text", "price"]) 

mask = ((df["room_type"]=="Entire home/apt") | (df["room_type"]=="Private room") & df["price_eur"].notna())

df["bathroom_type"]="Unknown"
df.loc[df["bathrooms_text"].notna(), "bathroom_type"]="Private"
df.loc[df["bathrooms_text"].str.contains("shared", case=False, na=False), "bathroom_type"]="Shared"

df["price_eur"]=df["price"].str.replace("$","").str.replace(",","")
df["price_eur"]=pd.to_numeric(df["price_eur"])

# Expected output:

# A wide table: 2 rows (room types) × 3 columns (bathroom types), containing median prices.
pivot_1 = (df
    .loc[mask]
    .pivot_table(
        index="room_type",
        columns="bathroom_type",
        values = "price_eur",
        aggfunc="median"
    )
    .round(1)
)
display(pivot_1)
# A second table with the same shape that gives the number of listings in each cell.
pivot_2 = (df
    .loc[mask]
    .pivot_table(
        index="room_type",
        columns="bathroom_type",
        values = "id",
        aggfunc="count",
        margins=True
    )
)
display(pivot_2)

# Sanity check (as code): the sum of the count table equals the number of priced listings of these two room types.
assert pivot_2.loc["All" , "All"]==df.loc[mask, "id"].count()

# Then, in up to 2 sentences: for private rooms, how much cheaper are listings with a shared bathroom (in the right unit), and which cell(s) are too small to trust?
# Private rooms with shared bathrooms are 89% cheaper in median terms than without (€103 vs €194.6). Not sure which cell is too small to trust - the main issue with this data is that entire homes have no information on the bathroom type at all, so the tables are only useful for private rooms.

# 1h20


bathroom_type,Private,Shared,Unknown
room_type,,,
Entire home/apt,226.0,NaN,245.0
Private room,194.0,103.0,113.0


bathroom_type,Private,Shared,Unknown,All
room_type,,,,
Entire home/apt,38128.0,NaN,3.0,38131
Private room,3569.0,1422.0,19.0,5010
All,41697.0,1422.0,22.0,43141
